# Interactive 2D-McDA viewer

This notebook opens a 2D-McDA NetCDF product, locates the corresponding CALIOP L1 file, loads the CALIOP attenuated-backscatter signals and Vertical Feature Mask (VFM), and displays twelve interactive panels with linked profile–altitude axes. Zooming or panning in one panel automatically updates all other panels.

Run the notebook from the `twod-mcda` environment. Copy `visualization/config.example.yaml` to `visualization/config.yaml`, then select the 2D-McDA product version with `product_directory`, identify the required output with `granule`, and set the displayed longitude bounds with `plot.longitude_range`. `l1_path` may remain `null` when the common processing configuration correctly describes the CALIOP archive.

In [ ]:
from datetime import datetime
from functools import reduce
from pathlib import Path
import json
import re

import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.feature.nightshade import Nightshade
import cmlidar
import holoviews as hv
import hvplot.xarray  # Register the hvPlot API for xarray objects.
import numpy as np
import xarray as xr
import yaml
import seaborn as sns
from bokeh.io import show
from bokeh.models import AllLabels, CustomJSHover, FixedTicker, HoverTool
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex
from matplotlib.ticker import MultipleLocator

from twod_mcda.caliop.constants import (
    CALIOP_L1_PRODUCT_TYPE,
    N_30M_BINS_PER_BIN_R2,
    N_30M_BINS_PER_BIN_R3,
    N_30M_BINS_PER_BIN_R4,
    N_BINS_R2,
    N_BINS_R3,
    N_BINS_R4,
    N_LASER_PULSES_PER_5km,
)
from twod_mcda.caliop.discovery import find_granule_file
from twod_mcda.caliop.geography import UTC_time_CALIPSO, get_monotical_lon
from twod_mcda.caliop.grids import unfold_vfm
from twod_mcda.caliop.hdf import HDF4Reader
from twod_mcda.caliop.reader import CALIOPRegularGridReader

hv.extension("bokeh")

In [ ]:
def find_project_root(start):
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not find the project root.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
VISUALIZATION_DIRECTORY = PROJECT_ROOT / "visualization"
VIEWER_CONFIG_PATH = VISUALIZATION_DIRECTORY / "config.yaml"
EXAMPLE_CONFIG_PATH = VISUALIZATION_DIRECTORY / "config.example.yaml"
DEFAULT_VFM_PATH = Path(
    "/home/ticjo/Documents/Pro/Recherche/codes/DATA/CALIOP/"
    "VFM.v5.00/2018/2018_08_31/"
    "CAL_LID_L2_VFM-Standard-V5-00.2018-08-31T21-33-53ZN.hdf"
)

# Fall back to the tracked example so that the notebook works immediately.
active_config_path = (
    VIEWER_CONFIG_PATH if VIEWER_CONFIG_PATH.exists() else EXAMPLE_CONFIG_PATH
)
with active_config_path.open() as stream:
    viewer_config = yaml.safe_load(stream) or {}


def resolve_project_path(value):
    if value is None:
        return None
    path = Path(value).expanduser()
    return path.resolve() if path.is_absolute() else (PROJECT_ROOT / path).resolve()


def required_config_value(mapping, key):
    value = mapping.get(key)
    if value is None or (isinstance(value, str) and not value.strip()):
        raise ValueError(f"Viewer configuration field {key!r} is required.")
    return value


def locate_2d_mcda_product(product_directory, granule):
    if not product_directory.is_dir():
        raise FileNotFoundError(product_directory)
    matches = sorted(product_directory.rglob(f"*{granule}.nc"))
    if not matches:
        raise FileNotFoundError(
            f"No 2D-McDA NetCDF matches granule {granule!r} below "
            f"{product_directory}."
        )
    if len(matches) > 1:
        formatted = "\n".join(f"- {path}" for path in matches)
        raise ValueError(
            f"Granule {granule!r} is ambiguous below {product_directory}:\n"
            f"{formatted}"
        )
    return matches[0].resolve()


PRODUCT_DIRECTORY = resolve_project_path(
    required_config_value(viewer_config, "product_directory")
)
PRODUCT_GRANULE = str(required_config_value(viewer_config, "granule"))
NETCDF_PATH = locate_2d_mcda_product(PRODUCT_DIRECTORY, PRODUCT_GRANULE)
L1_PATH = resolve_project_path(viewer_config.get("l1_path"))
VFM_PATH = resolve_project_path(viewer_config.get("vfm_path")) or DEFAULT_VFM_PATH
COMMON_CONFIG_PATH = resolve_project_path(
    viewer_config.get("common_config_path", "config/common.yaml")
)

plot_config = viewer_config.get("plot", {})
PLOT_WIDTH = int(plot_config.get("width", 500))
PLOT_HEIGHT = int(plot_config.get("height", 280))
configured_longitude_range = required_config_value(
    plot_config, "longitude_range"
)
if len(configured_longitude_range) != 2:
    raise ValueError("plot.longitude_range must contain exactly two bounds.")
PLOT_LONGITUDE_RANGE = tuple(float(value) for value in configured_longitude_range)
configured_altitude_range = plot_config.get("altitude_range")
ALTITUDE_RANGE = (
    tuple(configured_altitude_range)
    if configured_altitude_range is not None
    else None
)
print(f"Viewer configuration: {active_config_path}")
NETCDF_PATH

In [ ]:
MASK_VARIABLES = (
    "Parallel_Detection_Flags_532",
    "Perpendicular_Detection_Flags_532",
    "Detection_Flags_1064",
    "Composite_Detection_Flags",
)

SIGNAL_VARIABLES = (
    "Total_Attenuated_Backscatter_532",
    "Parallel_Attenuated_Backscatter_532",
    "Perpendicular_Attenuated_Backscatter_532",
    "Attenuated_Backscatter_1064",
)

GRANULE_PATTERN = re.compile(
    r"(\d{4}-\d{2}-\d{2}T\d{2}-\d{2}-\d{2}Z[DN])"
)
L1_FILENAME_PATTERN = re.compile(
    r"CAL_LID_L1-([^-]+)-(V\d+-\d+)\."
    r"(\d{4}-\d{2}-\d{2}T\d{2}-\d{2}-\d{2}Z[DN])\.hdf$"
)
VFM_FILENAME_PATTERN = re.compile(
    r"CAL_LID_L2_VFM-([^-]+)-(V\d+-\d+)\."
    r"(\d{4}-\d{2}-\d{2}T\d{2}-\d{2}-\d{2}Z[DN])\.hdf$"
)


def granule_id_from_product(path, dataset):
    for text in (path.name, str(dataset.attrs.get("id", ""))):
        match = GRANULE_PATTERN.search(text)
        if match is not None:
            return match.group(1)
    raise ValueError(
        "The CALIOP timestamp is absent from the filename and global id attribute."
    )


def locate_l1_file(product_path, dataset, config_path, explicit_path=None):
    if explicit_path is not None:
        path = Path(explicit_path).expanduser().resolve()
        if not path.exists():
            raise FileNotFoundError(path)
        return path

    with Path(config_path).open() as stream:
        config = yaml.safe_load(stream)

    granule_id = granule_id_from_product(product_path, dataset)
    granule_time = datetime.strptime(granule_id[:19], "%Y-%m-%dT%H-%M-%S")
    return find_granule_file(config, granule_time)


def l1_file_metadata(path):
    match = L1_FILENAME_PATTERN.fullmatch(path.name)
    if match is None:
        raise ValueError(f"Unrecognized CALIOP L1 filename: {path.name}")
    data_type, encoded_version, granule_id = match.groups()
    version = encoded_version.replace("-", ".", 1)
    return data_type, version, granule_id


def vfm_file_metadata(path):
    match = VFM_FILENAME_PATTERN.fullmatch(path.name)
    if match is None:
        raise ValueError(f"Unrecognized CALIOP VFM filename: {path.name}")
    data_type, encoded_version, granule_id = match.groups()
    version = encoded_version.replace("-", ".", 1)
    return data_type, version, granule_id


def normalize_product_dataset(dataset):
    missing = [name for name in MASK_VARIABLES if name not in dataset]
    if missing:
        raise KeyError(f"Variables missing from the 2D-McDA product: {missing}")

    rename = {}
    if "Profile_ID" in dataset.dims:
        rename["Profile_ID"] = "profile"
    if "Altitude" in dataset.dims:
        rename["Altitude"] = "altitude"
    normalized = dataset.rename(rename)
    return normalized.set_coords(
        [name for name in ("Latitude", "Longitude") if name in normalized]
    )


def matching_profile_indices(source_profiles, requested_profiles, source_name):
    source_profiles = np.asarray(source_profiles).squeeze().astype(np.int64)
    requested_profiles = np.asarray(requested_profiles).astype(np.int64)
    indices = np.searchsorted(source_profiles, requested_profiles)
    in_bounds = indices < source_profiles.size
    if not np.all(in_bounds) or not np.array_equal(
        source_profiles[indices], requested_profiles
    ):
        raise ValueError(f"Some Profile_ID values are absent from {source_name}.")
    if not np.all(np.diff(indices) == 1):
        raise ValueError(f"The selected {source_name} profiles are not consecutive.")
    return indices


def load_l1_track(l1_path):
    with HDF4Reader(str(l1_path)) as reader:
        return xr.Dataset(
            coords={
                "profile": reader.get_data("Profile_ID").astype(np.int64),
                "latitude": (
                    "profile",
                    reader.get_data("Latitude").astype(float),
                ),
                "longitude": (
                    "profile",
                    reader.get_data("Longitude").astype(float),
                ),
                "profile_utc_time": (
                    "profile",
                    reader.get_data("Profile_UTC_Time").astype(float),
                ),
            }
        )


def load_l1_signals(l1_path, requested_profiles, profile_indices, grid, altitude_count):
    data_type, version, granule_id = l1_file_metadata(l1_path)
    row_start = int(profile_indices[0])
    row_end = int(profile_indices[-1])

    with CALIOPRegularGridReader(
        product="L1",
        version=version,
        data_type=data_type or CALIOP_L1_PRODUCT_TYPE,
        granule_date=granule_id,
        grid=grid,
        slice_start=row_start,
        slice_end=row_end,
        slice_start_end_type="profindex",
        index30m_alt_max=altitude_count,
        folderpath=str(l1_path.parent),
    ) as reader:
        source_profiles = np.asarray(reader.get_data("Profile_ID")).astype(np.int64)
        if not np.array_equal(source_profiles, requested_profiles):
            raise ValueError("The L1 slice does not match the requested Profile_ID values.")
        source_altitudes = np.asarray(
            reader.get_data("Lidar_Data_Altitudes")
        ).astype(float)
        coordinates = {
            "profile": source_profiles,
            "altitude": source_altitudes,
            "latitude": (
                "profile", np.asarray(reader.get_data("Latitude")).astype(float)
            ),
            "longitude": (
                "profile", np.asarray(reader.get_data("Longitude")).astype(float)
            ),
            "profile_utc_time": (
                "profile",
                np.asarray(reader.get_data("Profile_UTC_Time")).astype(float),
            ),
        }
        arrays = {}
        for name in SIGNAL_VARIABLES:
            values = reader.get_data(name)
            expected_shape = (len(source_profiles), len(source_altitudes))
            if values.shape != expected_shape:
                raise ValueError(
                    f"Incompatible grid for {name}: {values.shape} versus "
                    f"{expected_shape}."
                )
            arrays[name] = (
                ("profile", "altitude"),
                np.ma.filled(values, np.nan),
            )

    return xr.Dataset(arrays, coords=coordinates)


def regularize_vfm_grid(values, native_altitudes):
    repeats = np.concatenate(
        [
            np.full(N_BINS_R4, N_30M_BINS_PER_BIN_R4),
            np.full(N_BINS_R3, N_30M_BINS_PER_BIN_R3),
            np.full(N_BINS_R2, N_30M_BINS_PER_BIN_R2),
        ]
    )
    if values.shape[1] != repeats.size or native_altitudes.size != repeats.size:
        raise ValueError("Unexpected VFM vertical grid.")
    step_30m = float(
        np.median(-np.diff(native_altitudes[-N_BINS_R2:]))
    )
    regular_altitudes = np.concatenate(
        [
            altitude
            + ((repeat - 1) / 2 - np.arange(repeat)) * step_30m
            for altitude, repeat in zip(native_altitudes, repeats)
        ]
    )
    regular_values = np.repeat(values, repeats, axis=1)
    return regular_values[:, ::-1], regular_altitudes[::-1]


def load_vfm_mask(vfm_path, requested_profiles, expected_granule_id):
    if not vfm_path.exists():
        raise FileNotFoundError(vfm_path)
    _, _, granule_id = vfm_file_metadata(vfm_path)
    if granule_id != expected_granule_id:
        raise ValueError(
            f"VFM granule {granule_id} does not match {expected_granule_id}."
        )

    with HDF4Reader(str(vfm_path)) as reader:
        all_profiles = reader.get_data("ssProfile_ID").astype(np.int64)
        profile_indices = matching_profile_indices(
            all_profiles, requested_profiles, "VFM"
        )
        row_start = int(profile_indices[0])
        row_end = int(profile_indices[-1])
        profile_count = row_end - row_start + 1
        record_start = row_start // N_LASER_PULSES_PER_5km
        record_end = row_end // N_LASER_PULSES_PER_5km
        dataset_shape = reader.get_sds_keys()[
            "Feature_Classification_Flags"
        ][1]
        folded = reader.get_data(
            "Feature_Classification_Flags",
            start=[record_start, 0],
            count=[record_end - record_start + 1, dataset_shape[1]],
            do_squeeze=False,
        )
        native_altitudes = reader.get_data("Lidar_Data_Altitudes").astype(float)
        source_latitudes = reader.get_data(
            "ssLatitude", start=[row_start, 0], count=[profile_count, 1]
        ).astype(float)
        source_longitudes = reader.get_data(
            "ssLongitude", start=[row_start, 0], count=[profile_count, 1]
        ).astype(float)
        source_utc_times = reader.get_data(
            "ssProfile_UTC_Time",
            start=[row_start, 0],
            count=[profile_count, 1],
        ).astype(float)

    unfolded = unfold_vfm(folded, put_in_all_alt_grid=False)
    local_start = row_start - record_start * N_LASER_PULSES_PER_5km
    local_end = local_start + profile_count
    values, source_altitudes = regularize_vfm_grid(
        unfolded[local_start:local_end], native_altitudes
    )
    return xr.DataArray(
        values,
        coords={
            "profile": requested_profiles,
            "altitude": source_altitudes,
            "latitude": ("profile", source_latitudes),
            "longitude": ("profile", source_longitudes),
            "profile_utc_time": ("profile", source_utc_times),
        },
        dims=("profile", "altitude"),
        name="Feature_Classification_Flags",
    )

In [ ]:
# decode_times=False prevents xarray from attempting to decode the TAI calendar.
product_ds = xr.open_dataset(NETCDF_PATH, decode_times=False)
product_ds = normalize_product_dataset(product_ds)
product_ds = product_ds.assign_coords(
    latitude=("profile", product_ds["Latitude"].values.squeeze()),
    longitude=("profile", product_ds["Longitude"].values.squeeze()),
    profile_utc_time=(
        "profile", product_ds["Profile_UTC_Time"].values.squeeze()
    ),
)

profiles = product_ds.coords["profile"].values
altitudes = product_ds.coords["altitude"].values
granule_id = granule_id_from_product(NETCDF_PATH, product_ds)

l1_path = locate_l1_file(
    NETCDF_PATH,
    product_ds,
    COMMON_CONFIG_PATH,
    explicit_path=L1_PATH,
)
l1_track = load_l1_track(l1_path)
l1_profile_indices = matching_profile_indices(
    l1_track.coords["profile"].values, profiles, "CALIOP L1"
)
l1_ds_333m = load_l1_signals(
    l1_path, profiles, l1_profile_indices, grid="333mx30m",
    altitude_count=len(altitudes),
)
l1_ds_5km = load_l1_signals(
    l1_path, profiles, l1_profile_indices, grid="5kmx60m",
    altitude_count=len(altitudes),
)
vfm_mask = load_vfm_mask(VFM_PATH, profiles, granule_id)

for source_name, source in (
    ("CALIOP L1", l1_ds_333m),
    ("CALIOP VFM", vfm_mask),
):
    if not np.array_equal(source.coords["profile"].values, profiles):
        raise ValueError(f"{source_name} Profile_ID alignment failed.")
    latitude_error = np.max(
        np.abs(
            source.coords["latitude"].values
            - product_ds.coords["latitude"].values
        )
    )
    longitude_error = np.max(
        np.abs(
            source.coords["longitude"].values
            - product_ds.coords["longitude"].values
        )
    )
    print(
        f"{source_name} alignment: max |Δlat|={latitude_error:.3e}°, "
        f"max |Δlon|={longitude_error:.3e}°"
    )

print(f"2D-McDA product : {NETCDF_PATH}")
print(f"CALIOP L1 file  : {l1_path}")
print(f"CALIOP VFM file : {VFM_PATH}")
print(f"Granule         : {granule_id}")
print(f"Profiles        : {profiles[0]} to {profiles[-1]} ({len(profiles)})")
print(
    f"L1 row indexes  : {l1_profile_indices[0]} to "
    f"{l1_profile_indices[-1]}"
)
print(f"Altitudes       : {altitudes[0]:.2f} to {altitudes[-1]:.2f} km")

In [ ]:
MAX_DETECTION_LEVEL = 5
CHANNEL_FLAG_VALUES = (0, 1, 2, 3, 4, 5, 250, 251, 252, 253, 254)
CHANNEL_FLAG_LABELS = (
    ["No detection"]
    + [f"Detection level {level}" for level in range(1, MAX_DETECTION_LEVEL + 1)]
    + [
        "Low confidence small strips",
        "Almost fully attenuated",
        "Fully attenuated",
        "Likely artifact",
        "Surface",
    ]
)
channel_palette = sns.cubehelix_palette(
    MAX_DETECTION_LEVEL,
    start=2,
    rot=1,
    hue=1.0,
    gamma=1.0,
    light=0.8,
    dark=0.2,
    reverse=True,
)
channel_palette.insert(0, (1.0, 1.0, 1.0))
channel_palette.extend(
    [
        (0.7, 0.0, 0.0),
        (0.8, 0.0, 0.0),
        (1.0, 0.0, 0.0),
        (0.8, 0.8, 0.8),
        (0.5, 0.0, 0.0),
    ]
)
CHANNEL_FLAG_COLORS = [to_hex(color) for color in channel_palette]

COMPOSITE_FLAG_VALUES = (1, 2, 3, 5, 7)
COMPOSITE_FLAG_LABELS = (
    "Clear air",
    "Atmospheric feature",
    "Low confidence",
    "Surface/Subsurface",
    "Fully attenuated",
)
COMPOSITE_FLAG_COLORS = (
    "#ffffff",
    "#00539c",
    to_hex((0.6, 0.6, 0.6)),
    to_hex((88 / 255, 41 / 255, 0 / 255)),
    to_hex((222 / 255, 41 / 255, 22 / 255)),
)
VFM_FEATURE_TYPES = tuple(range(8))
VFM_LABELS = (
    "Invalid",
    "Clear",
    "Cloud",
    "Tropospheric\nAerosol",
    "Stratospheric\nAerosol",
    "Surface",
    "Subsurface",
    "Fully\nAttenuated",
)
VFM_COLORS = (
    "#999999",
    "#77B3FB",
    "#FFFFF0",
    "#F3CF4F",
    "#FE9F6D",
    "#733C14",
    "#322115",
    "#DC332A",
)
BACKSCATTER_BOUNDS = [
    float(value) for value in cmlidar.cm.BACKSCATTER_DISCRETE_BOUNDS
]
BACKSCATTER_CMAP = cmlidar.cm.backscatter_18
# Include the under and over colors as equal-sized rectangular ranges.
BACKSCATTER_PALETTE = (
    [to_hex(BACKSCATTER_CMAP.get_under())]
    + [to_hex(BACKSCATTER_CMAP(index)) for index in range(BACKSCATTER_CMAP.N)]
    + [to_hex(BACKSCATTER_CMAP.get_over())]
)
BACKSCATTER_COLOR_COUNT = len(BACKSCATTER_PALETTE)


def format_latitude(value):
    return f"{abs(value):.2f}° {'S' if value < 0 else 'N'}"


def format_longitude(value):
    return f"{abs(value):.2f}° {'W' if value < 0 else 'E'}"


def plot_coordinates(array):
    return (
        np.asarray(array.coords["profile"].values),
        np.asarray(array.coords["altitude"].values),
        np.asarray(array.coords["latitude"].values),
        np.asarray(array.coords["longitude"].values),
    )


def longitude_profile_indices(source):
    longitudes = np.asarray(source.coords["longitude"].values)
    indices = [
        int(
            np.argmin(
                np.abs(((longitudes - bound + 180) % 360) - 180)
            )
        )
        for bound in PLOT_LONGITUDE_RANGE
    ]
    return min(indices), max(indices)


def longitude_profile_limits(source):
    profiles = np.asarray(source.coords["profile"].values)
    start, end = longitude_profile_indices(source)
    return float(profiles[start]), float(profiles[end])


def select_longitude_section(source):
    start, end = longitude_profile_indices(source)
    return source.isel(profile=slice(start, end + 1))


def latitude_longitude_ticks(array, count=6):
    profiles, _, latitudes, longitudes = plot_coordinates(array)
    indices = np.linspace(0, len(profiles) - 1, count, dtype=int)
    return [
        (
            float(profiles[index]),
            f"{format_latitude(latitudes[index])} / "
            f"{format_longitude(longitudes[index])}",
        )
        for index in indices
    ]


def altitude_limits(source_altitudes):
    if ALTITUDE_RANGE is not None:
        return ALTITUDE_RANGE
    return (float(np.min(source_altitudes)), float(np.max(source_altitudes)))


def format_scientific(value):
    exponent = int(np.floor(np.log10(value)))
    mantissa = value / 10**exponent
    superscripts = str.maketrans("-0123456789", "⁻⁰¹²³⁴⁵⁶⁷⁸⁹")
    return f"{mantissa:g}×10{str(exponent).translate(superscripts)}"


def colorbar_hook(labels, font_size="7pt", show_all=False):
    def hook(plot, element):
        colorbar = plot.handles.get("colorbar")
        if colorbar is None:
            return
        colorbar.ticker = FixedTicker(ticks=list(labels))
        colorbar.major_label_overrides = labels
        colorbar.major_label_text_font_size = font_size
        if show_all:
            colorbar.major_label_policy = AllLabels()
    return hook


def categorical_hover_formatter(labels):
    hover_labels = [str(label).replace("\n", " ") for label in labels]
    encoded_labels = json.dumps(hover_labels)
    return CustomJSHover(
        code=(
            f"const labels = {encoded_labels};\n"
            "const index = Math.round(value);\n"
            "return labels[index] ?? 'Unknown';"
        )
    )


def mask_plot(array, title, categories, category_labels, palette):
    array = select_longitude_section(array)
    plot_profiles, plot_altitudes, _, _ = plot_coordinates(array)
    values = array.transpose("altitude", "profile").values
    categories = np.asarray(categories)
    classes = np.full(values.shape, np.nan, dtype=float)
    for index, category in enumerate(categories):
        classes[values == category] = index

    labels = {
        float(index): label
        for index, label in enumerate(category_labels)
    }
    hover = HoverTool(
        tooltips=[
            ("Profile", "$x{0}"),
            ("Altitude", "$y{0.000} km"),
            ("Flag", "@flag{0}"),
            ("Detection class", "@image{custom}"),
        ],
        formatters={
            "@image": categorical_hover_formatter(category_labels)
        },
    )
    image = hv.Image(
        (plot_profiles, plot_altitudes, classes, values),
        kdims=["profile", "altitude"],
        vdims=["color_class", "flag"],
    )
    return image.opts(
        hv.opts.Image(
            cmap=list(palette),
            clim=(-0.5, len(categories) - 0.5),
            colorbar=True,
            clabel="Detection class",
            xticks=latitude_longitude_ticks(array),
            xlabel="Latitude / Longitude",
            ylabel="Altitude (km)",
            title=title,
            xlim=longitude_profile_limits(array),
            width=PLOT_WIDTH,
            height=PLOT_HEIGHT,
            ylim=altitude_limits(plot_altitudes),
            tools=[hover],
            hooks=[colorbar_hook(labels, font_size="6pt")],
        )
    )


def composite_mask_plot(array, title):
    array = select_longitude_section(array)
    plot_profiles, plot_altitudes, _, _ = plot_coordinates(array)
    raw_values = array.transpose("altitude", "profile").values
    finite = np.isfinite(raw_values)
    base_values = np.full(raw_values.shape, np.nan, dtype=float)
    base_values[finite] = np.bitwise_and(
        raw_values[finite].astype(np.uint8), 7
    )
    classes = np.full(raw_values.shape, np.nan, dtype=float)
    for index, category in enumerate(COMPOSITE_FLAG_VALUES):
        classes[base_values == category] = index

    labels = {
        float(index): label
        for index, label in enumerate(COMPOSITE_FLAG_LABELS)
    }
    hover = HoverTool(
        tooltips=[
            ("Profile", "$x{0}"),
            ("Altitude", "$y{0.000} km"),
            ("Composite flag", "@composite_flag{0}"),
            ("Composite class", "@image{custom}"),
        ],
        formatters={
            "@image": categorical_hover_formatter(COMPOSITE_FLAG_LABELS)
        },
    )
    image = hv.Image(
        (plot_profiles, plot_altitudes, classes, base_values, raw_values),
        kdims=["profile", "altitude"],
        vdims=["color_class", "base_class", "composite_flag"],
    )
    return image.opts(
        hv.opts.Image(
            cmap=list(COMPOSITE_FLAG_COLORS),
            clim=(-0.5, len(COMPOSITE_FLAG_VALUES) - 0.5),
            colorbar=True,
            clabel="Composite class",
            xticks=latitude_longitude_ticks(array),
            xlabel="Latitude / Longitude",
            ylabel="Altitude (km)",
            title=title,
            xlim=longitude_profile_limits(array),
            width=PLOT_WIDTH,
            height=PLOT_HEIGHT,
            ylim=altitude_limits(plot_altitudes),
            tools=[hover],
            hooks=[colorbar_hook(labels, font_size="6pt")],
        )
    )


def vfm_plot(array, title):
    array = select_longitude_section(array)
    plot_profiles, plot_altitudes, _, _ = plot_coordinates(array)
    raw_values = array.transpose("altitude", "profile").values
    finite = np.isfinite(raw_values)
    feature_types = np.full(raw_values.shape, np.nan, dtype=float)
    feature_types[finite] = np.bitwise_and(
        raw_values[finite].astype(np.uint16), 7
    )
    labels = {
        float(index): label for index, label in enumerate(VFM_LABELS)
    }
    hover = HoverTool(
        tooltips=[
            ("Profile", "$x{0}"),
            ("Altitude", "$y{0.000} km"),
            ("VFM feature type", "@image{custom}"),
            ("Feature classification flag", "@vfm_flag{0}"),
        ],
        formatters={
            "@image": categorical_hover_formatter(VFM_LABELS)
        },
    )
    image = hv.Image(
        (plot_profiles, plot_altitudes, feature_types, raw_values),
        kdims=["profile", "altitude"],
        vdims=["feature_type", "vfm_flag"],
    )
    return image.opts(
        hv.opts.Image(
            cmap=list(VFM_COLORS),
            clim=(-0.5, len(VFM_FEATURE_TYPES) - 0.5),
            colorbar=True,
            clabel="VFM feature type",
            xticks=latitude_longitude_ticks(array),
            xlabel="Latitude / Longitude",
            ylabel="Altitude (km)",
            title=title,
            xlim=longitude_profile_limits(array),
            width=PLOT_WIDTH,
            height=PLOT_HEIGHT,
            ylim=altitude_limits(plot_altitudes),
            tools=[hover],
            hooks=[colorbar_hook(labels, font_size="6pt", show_all=True)],
        )
    )


def backscatter_plot(array, title):
    array = select_longitude_section(array)
    plot_profiles, plot_altitudes, _, _ = plot_coordinates(array)
    values = array.transpose("altitude", "profile").values
    classes = np.full(values.shape, np.nan, dtype=float)
    valid = np.isfinite(values)
    classes[valid] = np.digitize(values[valid], bins=BACKSCATTER_BOUNDS)
    labels = {
        index + 0.5: format_scientific(bound)
        for index, bound in enumerate(BACKSCATTER_BOUNDS)
    }
    hover = HoverTool(
        tooltips=[
            ("Profile", "$x{0}"),
            ("Altitude", "$y{0.000} km"),
            ("β′", "@backscatter{0.000000e} km⁻¹ sr⁻¹"),
        ]
    )
    image = hv.Image(
        (plot_profiles, plot_altitudes, classes, values),
        kdims=["profile", "altitude"],
        vdims=["color_class", "backscatter"],
    )
    return image.opts(
        hv.opts.Image(
            cmap=BACKSCATTER_PALETTE,
            clim=(-0.5, BACKSCATTER_COLOR_COUNT - 0.5),
            colorbar=True,
            clabel="β′ (km⁻¹ sr⁻¹)",
            xticks=latitude_longitude_ticks(array),
            xlabel="Latitude / Longitude",
            ylabel="Altitude (km)",
            title=title,
            bgcolor="black",
            xlim=longitude_profile_limits(array),
            width=PLOT_WIDTH,
            height=PLOT_HEIGHT,
            ylim=altitude_limits(plot_altitudes),
            tools=[hover],
            hooks=[colorbar_hook(labels)],
        )
    )

In [ ]:
plots = [
    # Row 1: total 532 nm signal, VFM, and composite mask.
    backscatter_plot(
        l1_ds_5km["Total_Attenuated_Backscatter_532"],
        "Total Attenuated Backscatter 532 nm — 5 km × 60 m",
    ),
    vfm_plot(
        vfm_mask,
        "Vertical Feature Mask (VFM) — 333 m × 30 m",
    ),
    composite_mask_plot(
        product_ds["Composite_Detection_Flags"],
        "Composite Detection Flags — 333 m × 30 m",
    ),
    # Row 2: parallel 532 nm signal and mask.
    backscatter_plot(
        l1_ds_5km["Parallel_Attenuated_Backscatter_532"],
        "Parallel Attenuated Backscatter 532 nm — 5 km × 60 m",
    ),
    backscatter_plot(
        l1_ds_333m["Parallel_Attenuated_Backscatter_532"],
        "Parallel Attenuated Backscatter 532 nm — 333 m × 30 m",
    ),
    mask_plot(
        product_ds["Parallel_Detection_Flags_532"],
        "Parallel Detection Flags 532 nm — 333 m × 30 m",
        CHANNEL_FLAG_VALUES,
        CHANNEL_FLAG_LABELS,
        CHANNEL_FLAG_COLORS,
    ),
    # Row 3: perpendicular 532 nm signal and mask.
    backscatter_plot(
        l1_ds_5km["Perpendicular_Attenuated_Backscatter_532"],
        "Perpendicular Attenuated Backscatter 532 nm — 5 km × 60 m",
    ),
    backscatter_plot(
        l1_ds_333m["Perpendicular_Attenuated_Backscatter_532"],
        "Perpendicular Attenuated Backscatter 532 nm — 333 m × 30 m",
    ),
    mask_plot(
        product_ds["Perpendicular_Detection_Flags_532"],
        "Perpendicular Detection Flags 532 nm — 333 m × 30 m",
        CHANNEL_FLAG_VALUES,
        CHANNEL_FLAG_LABELS,
        CHANNEL_FLAG_COLORS,
    ),
    # Row 4: 1064 nm signal and mask.
    backscatter_plot(
        l1_ds_5km["Attenuated_Backscatter_1064"],
        "Attenuated Backscatter 1064 nm — 5 km × 60 m",
    ),
    backscatter_plot(
        l1_ds_333m["Attenuated_Backscatter_1064"],
        "Attenuated Backscatter 1064 nm — 333 m × 30 m",
    ),
    mask_plot(
        product_ds["Detection_Flags_1064"],
        "Detection Flags 1064 nm — 333 m × 30 m",
        CHANNEL_FLAG_VALUES,
        CHANNEL_FLAG_LABELS,
        CHANNEL_FLAG_COLORS,
    ),
]

## Granule and observed section

In [ ]:
def plot_granule_map(granule_id, granule_track, section):
    section_utc = section.coords["profile_utc_time"].values
    start_utc_time = UTC_time_CALIPSO(float(section_utc[0]))
    end_utc_time = UTC_time_CALIPSO(float(section_utc[-1]))
    start_date = datetime.strptime(start_utc_time, "%Y-%m-%d %H:%M:%S")
    end_date = datetime.strptime(end_utc_time, "%Y-%m-%d %H:%M:%S")

    granule_latitudes = granule_track.coords["latitude"].values
    granule_longitudes = get_monotical_lon(
        granule_track.coords["longitude"].values
    )
    section_latitudes = section.coords["latitude"].values
    section_longitudes = get_monotical_lon(
        section.coords["longitude"].values
    )
    middle = section_latitudes.size // 2
    latitude_center = float(section_latitudes[middle])
    longitude_center = float(section_longitudes[middle])

    figure = plt.figure(figsize=(6, 6))
    projection = ccrs.Orthographic(longitude_center, latitude_center)
    axis = plt.axes(projection=projection)
    axis.stock_img()
    axis.coastlines(rasterized=True)
    axis.add_feature(cfeature.LAKES, edgecolor="black", facecolor="none")
    axis.add_feature(Nightshade(start_date, alpha=0.2), rasterized=True)
    axis.add_feature(Nightshade(end_date, alpha=0.2), rasterized=True)

    minor_grid = axis.gridlines(
        color="black", linewidth=0.5, linestyle="--", alpha=0.2
    )
    minor_grid.xlocator = MultipleLocator(10)
    minor_grid.ylocator = MultipleLocator(10)
    major_grid = axis.gridlines(
        color="black", linewidth=1.0, linestyle="-", alpha=0.2
    )
    major_grid.xlocator = MultipleLocator(30)
    major_grid.ylocator = MultipleLocator(30)
    axis.set_global()

    axis.plot(
        granule_longitudes,
        granule_latitudes,
        color="blue",
        linewidth=4,
        alpha=0.1,
        transform=ccrs.PlateCarree(),
        rasterized=True,
        label="Complete CALIOP granule",
    )
    section_color = "#d92409" if granule_id.endswith("ZN") else "#a81c07"
    axis.plot(
        section_longitudes,
        section_latitudes,
        color=section_color,
        linewidth=4,
        transform=ccrs.PlateCarree(),
        rasterized=True,
        label="2D-McDA section",
    )
    axis.legend(loc="lower center")
    axis.set_title(
        f"{granule_id}\n{start_utc_time} – {end_utc_time}",
        weight="bold",
    )
    figure.subplots_adjust(left=0.02, bottom=0.02, right=0.98, top=0.90)
    return figure


plot_section = select_longitude_section(product_ds)
plot_profiles = plot_section.coords["profile"].values
plot_longitudes = plot_section.coords["longitude"].values
print(f"Granule: {granule_id}")
print(
    f"Plotted longitude section: {plot_longitudes[0]:.2f}° to "
    f"{plot_longitudes[-1]:.2f}°; Profile_ID {plot_profiles[0]} to "
    f"{plot_profiles[-1]} ({len(plot_profiles)} profiles)"
)
granule_map = plot_granule_map(granule_id, l1_track, plot_section)

## Display

All twelve panels share their `profile` and `altitude` ranges. Use the mouse wheel or Box Zoom tool in any panel; all other panels update automatically. The Reset button restores the initial extent.

In [ ]:
viewer = reduce(lambda left, right: left + right, plots).cols(3)
viewer = viewer.opts(
    hv.opts.Layout(
        shared_axes=True,
        merge_tools=True,
    )
)
viewer_plot = hv.render(viewer, backend="bokeh")
show(viewer_plot)

In [ ]:
# Run this cell when the exploration is complete.
product_ds.close()
plt.close(granule_map)